In [ ]:
try:
    import firedrake
except ImportError:
    !wget "https://fem-on-colab.github.io/releases/firedrake-install-release-real.sh" -O "/tmp/firedrake-install.sh" && bash "/tmp/firedrake-install.sh"
    import firedrake

try:
    import irksome
except ImportError:
    !python3 -m pip install --no-dependencies git+https://github.com/firedrakeproject/Irksome.git#egg=Irksome
    import irksome

In [ ]:
# Code in this cell makes plots appear an appropriate size and resolution in the browser window
%matplotlib widget
%config InlineBackend.figure_format = 'svg'

import matplotlib.pyplot as plt

plt.rcParams['figure.figsize'] = (11, 6)

This notebook covers an example of variational data assimilation in Firedrake, specifically "Strong Constraint 4DVar".
This is an inverse problem for the initial condition $x_{0}$ of a timeseries which best matches some observations of the system $y_{i}$ at times $t_{i}$, constrained by a (discretised) PDE model.
This is commonly used in operational weather forecasting to assimilate atmospheric observations into the numerical model.

The cost function to be minimised is:
$$
\mathcal{J}(x_{0}) = \|x_{0} - x_{b}\|^{2}_{B^{-1}} + \sum^{N}_{i=0}\|\mathcal{G}_{i}(x_{0}) - y_{i}\|^{2}_{R^{-1}}
$$

$x_{b}$ is the prior estimate of $x_{0}$ (often called the "background" state in the 4DVar literature).
The operator $\mathcal{G}$ is the composition of the PDE model $\mathcal{M}_{i}$ that propagates the state from $t_{0}$ to $t_{i}$, and the observation operator $\mathcal{H}$ which maps from a state $x$ to an observation $y$. $\mathcal{H}$ is often low rank, for example point observations, and in practical applications may also be nonlinear.

$$
\mathcal{G}_{i}(x) = \mathcal{H}_{i}(\mathcal{M}_{i}(x)),
\quad
\mathcal{M}_{i}: x(t=0) \to x(t=t_{i})
$$

The data is assumed to include Gaussian noise with covariance matrices $B$ and $R$ for the initial condition and the observations respectively.
The norms in the objective functional are weighted by these matrices to encode the underlying uncertainty.
$$
x = x_{\textrm{true}} + z_{b},
\quad
z_{b} \sim \mathcal{N}(0, B)
$$

$$
\mathcal{G}(x) = \mathcal{G}(x_{\textrm{true}}) + z_{r},
\quad
z_{r} \sim \mathcal{N}(0, R)
$$

To solve the optimisation problem we use a Gauss-Newton method, where at each iteration we solve for the update $\delta x$.
$$
J(\delta x) = \|\delta x - b\|^{2}_{B^{-1}} + \sum^{N}_{i=0}\|G_{i}\delta x - d_{i}\|^{2}_{R^{-1}}
$$
where $b$ and $d_{i}$ are misfit vectors.

This amounts to solving the following (SPD) lineae system, where $g$ is the gradient at the current iterate.
$$
S\delta x = g
\quad\to\quad
\left(B^{-1} + \sum G_{i}^{T}R^{-1}G_{i}\right)\delta x = g
$$

In this example we will solve the advection diffusion equation in 2D for a temperature $T$, with a nonlinear diffusivity $\mu=\mu(T)$, advective velocity $u$, and forcing $f$:
$$
\partial_{t}T + u\cdot\nabla T + \nabla\cdot\left(\mu(T)\nabla T\right) = f
$$

We need a few things in addition to the main Firedrake library.
First, the components from `firedrake.adjoint` which will give us access to automatically generated adjoint/tlm models via automatic differentiation.
Second, the `TAOSolver` from pyadjoint to solve the optimisation problem.
Thirdly, we will use [Irksome](https://www.firedrakeproject.org/Irksome/) for the time integration of the PDE.

In [ ]:
import numpy as np
from firedrake import *
from firedrake.adjoint import *
from pyadjoint import TAOSolver
from irksome import TimeStepper, Dt, GaussLegendre
Print = PETSc.Sys.Print

We first define some problem parameters, elements on each edge of the 2D mesh `nx`, number of timesteps `nt`, how often we take observations of the system `obs_freq`, and the timestep `dt`.
The velocity field will be a vortex with strength `vel0` and a background horizontal flow with strength `hvel0`.

The mesh is translated to be centred on (0,0), and the temperature will live in CG1, the space of continuous piecewise linear coefficients, and the velocity will live in a vector CG2 space.

In [ ]:
nx = 80
nt = 20
obs_freq = 2
dt_f = 4e-2

vel0 = Constant(1)
hvel0 = Constant(0.4)
reynolds = Constant(300)

h = 1/nx
cfl_v = float((vel0+hvel0)*dt_f/h)
cfl_nu = float((1/reynolds)*dt_f/h**2)

Print(f"T= {nt*dt_f:.2f} . {cfl_v= :.2f} . {cfl_nu= :.2f}")

mesh = PeriodicUnitSquareMesh(nx, nx, quadrilateral=True)
mesh.coordinates.interpolate(mesh.coordinates - as_vector([0.5, 0.5]))
x, y = SpatialCoordinate(mesh)

V = FunctionSpace(mesh, "CG", 1)
Vv = VectorFunctionSpace(mesh, "CG", 2)
Real = FunctionSpace(mesh, "R", 0)

The vortical part of the velocity field is based on an Oseen-Lamb vortex which decays towards the periodic boundaries.
The streamlines show the pathlines of the full velocity field including the horizontal background flow.

In [ ]:
# advecting velocity
theta = atan2(y, x)
r = sqrt(x*x + y*y)

rc = Constant(0.25)
eps = Constant(1e-8)
rmax = Constant(0.48)

vel_ref = Constant(vel0)
rceil = conditional(r < rmax, r, rmax)
vtheta = ((rmax - rceil)/(rmax - rc))*(vel_ref/(r/rc + eps))*(1 - exp(-(r*r)/(rc*rc)))
vel = Function(Vv, name="velocity").interpolate(
    + hvel0*as_vector([1, 0])
    + vtheta*as_vector([-sin(theta), cos(theta)])
)

fig, axes = plt.subplots()
streamlines = streamplot(vel, resolution=1/40, seed=0, axes=axes)
fig.colorbar(streamlines, ax=axes, fraction=0.04)
axes.set_aspect("equal")
axes.set_title("Velocity")
fig.show()

We will specify the true initial condition $x_{\textrm{true}}$ and prior $x_{b}$ as perturbations of a common reference initial condition $\hat{x}$ by some noise generated from $B$.

$$
x_{\textrm{true}} = \hat{x} + z_{b,0},
\quad
x_{b} = \hat{x} + z_{b,1}
$$

The reference initial condition is a starfish with a smooth decay at the ends of the fingers.

In [ ]:
# initial condition
ic_min = Constant(0.1)
ic_max = Constant(0.5)
ic_radius = 0.5*((ic_max + ic_min) + (ic_max - ic_min)*sin(5*theta))
dist = conditional(r/ic_radius < 1, r/ic_radius, 1)
ic_expr = 0.5*(1 + cos(pi*dist**3))
ic_reference = Function(V, name="ic_reference").interpolate(ic_expr)

fig, axes = plt.subplots()
contours = tricontourf(ic_reference, 30, axes=axes)
fig.colorbar(contours, ax=axes, fraction=0.04)
axes.set_aspect("equal")
axes.set_title("Reference IC")
fig.show()

Now we define the discretised PDE model for $\mathcal{M}_{i}$.

The diffusivity will depend cubically on the temperature:
$$
\mu(T) = \mu_{0} + \mu'T^3
$$
Which we write as a linear combination of the viscosity $1/Re$ at $T=1$ and $\alpha/Re$ at $T=0$:
$$
\mu_{0} = \alpha/Re, \quad \mu' = (1 - \alpha)/Re
$$

For the time derivative we use the 1 stage Gauss-Legendre scheme from Irksome, which is just the implicit midpoint rule.

In [ ]:
alpha = Constant(0.3)
mu_pow = 3

def mu(T):
    return (1/reynolds)*(alpha + (1 - alpha)*T**mu_pow)

rhs = Function(V, name="source").assign(hvel0*ic_reference)

dt = Function(Real).assign(dt_f)
t = Function(Real).assign(0.)

T = Function(V, name="Temperature")
v = TestFunction(V)
F = (
    inner(Dt(T), v)*dx
    + inner(dot(vel, grad(T)), v)*dx
    + inner(mu(T)*grad(T), grad(v))*dx
    - inner(rhs, v)*dx
)

parameters = {
    'snes_type': 'newtonls',
    'snes_rtol': 1e-8,
    'ksp_type': 'preonly',
    'mat_type': 'aij',
    'pc_type': 'lu',
    'pc_factor_mat_solver_type': 'mumps',
    'pc_factor_reuse_ordering': None,
}

fcp = {'max_quadrature_degree': 6}

stepper = TimeStepper(
    F, GaussLegendre(1), t, dt, T,
    solver_parameters=parameters,
    options_prefix="irk",
    form_compiler_parameters=fcp,
)

The covariance operator is an autoregressive covariance operator of order $m$ (a particular type of Matern covariance operator), which approach the Gaussian covariance as $m\to\infty$.
These are particularly nice in a finite element setting because the action of $B$ on a vector $x$ can be written as $m$ implicit Euler steps of a standard heat equation.
The diffusion strength of the heat equation is determined by the correlation lengthscale $L$, and the whole thing is scaled by the standard deviation $\sigma$.

They can also be used to efficiently generate random samples from $B$ and calculate the weighted norm $\|x\|_{B^{-1}}$ by applying only $m/2$ implicit Euler steps.
We plot the random noise vectors used to perturb the true and prior initial conditions from $\hat{x}$.

More detail in the firedrake docs for the
[autoregressive covariance](https://www.firedrakeproject.org/firedrake.adjoint.html#firedrake.adjoint.covariance_operator.AutoregressiveCovariance)
and the
[white noise generation](https://www.firedrakeproject.org/firedrake.adjoint.html#firedrake.adjoint.covariance_operator.WhiteNoiseGenerator)

In [ ]:
B = AutoregressiveCovariance(V, L=0.2, sigma=1e-1, m=4, seed=17)
z_truth = B.sample()
z_prior = B.sample()

ic_truth = Function(V, name="ic_truth").assign(ic_reference + z_truth)
ic_prior = Function(V, name="ic_prior").assign(ic_reference + z_prior)

fig, axes = plt.subplots(ncols=2, sharex=True, sharey=True)
contours = tricontourf(z_truth, 30, axes=axes[0])
fig.colorbar(contours, ax=axes[0], fraction=0.03)
axes[0].set_aspect("equal")
axes[0].set_title("True IC perturbation")

contours = tricontourf(z_prior, 30, axes=axes[1])
fig.colorbar(contours, ax=axes[1], fraction=0.03)
axes[1].set_aspect("equal")
axes[1].set_title("Prior IC perturbation")
fig.show()

The next piece of the puzzle is the observation operator $\mathcal{H}$.
For this example we just use point observations by interpolating onto a `VertexOnlyMesh` ([manual page](https://www.firedrakeproject.org/point-evaluation.html)).
In more complicated problems we might interpolate into higher dimensional meshes (e.g. a line mesh), or observe a nonlinear function of the state (e.g. the energy).

In [ ]:
## Observation locations
nstations_1D = 25
locations_1D = -0.5 + np.linspace(
    1/(nstations_1D+1), 1, nstations_1D, endpoint=False)
locations = np.zeros((nstations_1D**2, 2))
for i in range(nstations_1D):
    offset = i*nstations_1D
    locations[offset:offset+nstations_1D, 0] = locations_1D[i]
    locations[offset:offset+nstations_1D, 1] = locations_1D

stations = VertexOnlyMesh(mesh, locations)
U = FunctionSpace(stations, "DG", 0)

def H(x):
    return assemble(interpolate(x, U))

The observations are considered independent, i.e. a diagonal covariance operator, which is achieved with the "$0$-th order" autoregressive covariance.

In [ ]:
R = AutoregressiveCovariance(U, L=0, sigma=1e-3, m=0, seed=6)

For this example we don't have any real-world observational data, so we generate our own synthetic data by integrating the "true" initial condition and adding noise to the observations which is consistent with our original definition of the variational problem.

$$
x_{\textrm{true}} = \hat{x} + z_{b}
$$

$$
y_{i} = \mathcal{G}_{i}(x_{\textrm{true}}) + z_{r} = \mathcal{H}_{i}(\mathcal{M}_{i}(x_{\textrm{true}})) + z_{r},
\quad
z_{r} \sim \mathcal{N}(0,R)
$$

We plot the initial and final conditions of the "true" trajectory.

In [ ]:
# Synthetic data
t.assign(0.)
T.assign(ic_truth)

y = [Function(U).assign(H(T) + R.sample())]
for i in range(nt):
    stepper.stages.zero()
    stepper.advance()
    t.assign(t + dt)
    if ((i+1) % obs_freq) == 0:
        y.append(Function(U).assign(H(T) + R.sample()))

mu_end = Function(V, name="diffusivity").interpolate(mu(T))

fig, axes = plt.subplots(ncols=3, sharey=True)
contours = tricontourf(ic_truth, 30, axes=axes[0])
fig.colorbar(contours, ax=axes[0], fraction=0.02)
axes[0].set_aspect("equal")
axes[0].set_title("True initial condition")

contours = tricontourf(T, 30, axes=axes[1])
fig.colorbar(contours, ax=axes[1], fraction=0.02)
axes[1].set_aspect("equal")
axes[1].set_title("True final condition")

contours = tricontourf(mu_end, 30, axes=axes[2])
fig.colorbar(contours, ax=axes[2], fraction=0.02)
axes[2].set_aspect("equal")
axes[2].set_title("Final diffusivity")
fig.show()

Now we "record" our forward problem.
Once we switch on annotations, pyadjoint and Firedrake will record (symbolically) every operation carried out on a tape.
The `ReducedFunctional` class abstracts the definition of $\mathcal{J}(x)$, based on a functional value $J$, a control value $x$, and a tape which records all operations needed to calculate $J$ from $x$.

Once the `ReducedFunctional` is created we can re-evaluate it at different values of the control $\mathcal{J}(w)$, and we have immediate access to the derivative, tangent linear, and Hessian models, which are calculated by running forward or backward through the tape and using the symbolic record to automatically calculate the derivatives of J using UFL.

In [ ]:
# initial guess of prior
t.assign(0)
ic = ic_prior.copy(deepcopy=True)

idx = 0
continue_annotation()
with set_working_tape() as tape:
    J = B.norm(ic - ic_prior)
    T.assign(ic)
    J += R.norm((H(T) - y[idx]))
    idx += 1
    for i in range(nt):
        stepper.stages.zero()
        stepper.advance()
        t.assign(t + dt)
        if ((i+1) % obs_freq) == 0:
            J += R.norm((H(T) - y[idx]))
            idx += 1
    Jhat = ReducedFunctional(J, Control(ic), tape=tape)
pause_annotation()

We can now create a `TAOSolver` for the optimisation problem of minimising $\mathcal{J}$ over the control $x$.
We specify the Newton solver using an Options dictionary.

Remember that we are solving the following linear system:
$$
S\delta x = g,
\quad
S = B^{-1} + \sum G_{i}^{T}R^{-1}G_{i}
$$
It is common to precondition this system with the covariance operator $B$, because we end up with a matrix that is the identity plus a low rank (but usually very poorly conditioned) perturbation.
$$
B^{T/2}SB^{1/2} = I + \sum B^{T/2}G_{i}^{T}R^{-1}G_{i}B^{1/2}
$$
To do this with the `TAOSolver` we have to provide a `CovarianceMat` built from $B$ as the preconditioning matrix. We need to be careful with how we specify this: the matrix action is the _inverse_ of $B$.
The `firedrake.CovariancePC` python preconditioner will then apply the _inverse_ of the `CovarianceMat`, which is the _action_ of $B$ (which requires the implicit Euler steps of the diffusion equation).

In [ ]:
tao_parameters = {
    'tao_view': ':tao_view.log',
    'tao_monitor': None,
    'tao_gttol': 3e-2,
    'tao_type': 'nls',
    'tao_ls_type': 'unit',
    'tao_nls': {
        'ksp_monitor': None,
        'ksp_converged_reason': None,
        'ksp_convergence_test': 'skip',
        'ksp_max_it': 10,
        'ksp_type': 'cg',
        'pc_type': 'python',
        'pc_python_type': 'firedrake.CovariancePC',
    },
}

tao = TAOSolver(
    MinimizationProblem(Jhat),
    Pmat=CovarianceMat(B, 'inverse'),
    parameters=tao_parameters,
    options_prefix="")

Now we have everything set up we can solve the system.
This should take a couple of minutes, and we see a reduction in the error from around $25\%$ down to around $1\%$.

In [ ]:
ic_opt = tao.solve()
ic_opt.rename("ic_opt")
Print(f"{errornorm(ic_truth, ic_prior) = :.2e}")
Print(f"{errornorm(ic_truth, ic_opt)   = :.2e}")

Finally, we plot the reference, true, prior, and optimised initial conditions.
We can see that the optimised solution clearly approximates the true solution very well.

In [ ]:
fig, axes = plt.subplots(nrows=2, ncols=2, sharex=True, sharey=True)

contours = tricontourf(ic_reference, 30, axes=axes[0,0])
fig.colorbar(contours, ax=axes[0,0], fraction=0.02)
axes[0,0].set_aspect("equal")
axes[0,0].set_title("Reference IC")

contours = tricontourf(ic_truth, 30, axes=axes[0,1])
fig.colorbar(contours, ax=axes[0,1], fraction=0.02)
axes[0,1].set_aspect("equal")
axes[0,1].set_title("True IC")

contours = tricontourf(ic_prior, 30, axes=axes[1,0])
fig.colorbar(contours, ax=axes[1,0], fraction=0.02)
axes[1,0].set_aspect("equal")
axes[1,0].set_title("Prior IC")

contours = tricontourf(ic_opt, 30, axes=axes[1,1])
fig.colorbar(contours, ax=axes[1,1], fraction=0.02)
axes[1,1].set_aspect("equal")
axes[1,1].set_title("Optimised IC")

fig.show()